# 04 · Volatility, regimes and the FX overlay

Loads `data/processed/weekly.csv` and `daily_factor.csv`, saves `data/processed/overlay.csv`.

The direction of the factor proved unpredictable (notebook 03). This notebook uses the models
for what is predictable, next week's volatility, and builds the FX overlay for a Swiss investor
holding 100,000 CHF of S&P 500.

Steps (see PLAN.md, sections 5c and 5d):
1. Load the data and set aside the holdout (2024 onward is not used here)
2. PCA: is the basket really the main common force of the 14 pairs?
3. Inputs and target for the volatility models
4. Four volatility forecasts, walk-forward: naive, linear regression, GARCH, Gaussian process
5. Forecast accuracy: which model beats the naive forecast?
6. Market regimes: GMM, compared with k-means
7. The two FX overlays: trend rule and model
8. The four portfolios in CHF
9. Crisis table and performance table
10. Checks

In [ ]:
import numpy as np
import pandas as pd
from arch import arch_model
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from sklearn.linear_model import LinearRegression
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

import helpers

np.random.seed(helpers.SEED)

## 1. Load the data and set aside the holdout

In [ ]:
weekly = pd.read_csv(f"{helpers.PROCESSED}/weekly.csv", index_col=0, parse_dates=True)
daily = pd.read_csv(f"{helpers.PROCESSED}/daily_factor.csv", index_col=0, parse_dates=True).iloc[:, 0]

# The holdout (2024 onward) stays untouched until the very end of the project
weekly = weekly[weekly.index < helpers.HOLDOUT_START]
daily = daily[daily.index < helpers.HOLDOUT_START]

factor = weekly["factor"]
rv = weekly["rv"]
print("Weeks:", len(weekly), "to", weekly.index[-1].date(), "| days:", len(daily))

## 2. PCA: is the basket the main common force?

In [ ]:
# PCA finds, on its own, the combinations of pairs that explain most of the movement.
# If the first one looks like our equal-weight basket, the factor is confirmed.
pair_cols = [f"ret_{p}" for p in helpers.ALL_PAIRS]
pca = PCA().fit(weekly[pair_cols])

share = pca.explained_variance_ratio_
print(f"First component explains {share[0]:.0%} of all movement (second: {share[1]:.0%})")

# Its weights, sign-adjusted so the average weight is positive
w = pd.Series(pca.components_[0], index=helpers.ALL_PAIRS)
w = w * np.sign(w.mean())
print("\nWeights of the first component (the basket uses 1/14 = 0.07 everywhere):")
print(w.round(2).to_string())

same_sign = (w > 0).sum()
print(f"\n{same_sign} of 14 weights are positive")
pc1 = weekly[pair_cols].values @ (w / w.abs().sum()).values
corr = np.corrcoef(pc1, weekly["factor_spot"])[0, 1]
print(f"Correlation of the first component with the basket: {corr:.3f}")

## 3. Inputs and target for the volatility models

In [ ]:
# We work with the logarithm of volatility: it makes big and small weeks comparable.
# Inputs known at the Friday close of week t; target is next week's volatility.
log_rv = np.log(rv)
X = pd.DataFrame({
    "log_rv_1w": log_rv,
    "log_rv_4w": np.log(rv.rolling(4).mean()),
    "log_rv_13w": np.log(rv.rolling(13).mean()),
    "log_vix": np.log(weekly["VIX"]),
})
y = log_rv.shift(-1).rename("log_rv_next")

valid = X.dropna().index.intersection(y.dropna().index)
X, y = X.loc[valid], y.loc[valid]
print("Rows:", len(X), "from", X.index[0].date(), "to", X.index[-1].date())

## 4. Four volatility forecasts, walk-forward

In [ ]:
# For each test year (2010 to 2023), every model is estimated on the years before it.
# All four produce, at each Friday, a forecast of NEXT week's volatility (annualised).

test_years = range(helpers.FIRST_TEST_YEAR, 2024)
fc_naive, fc_reg, fc_garch, fc_gp, fc_gp_std = [], [], [], [], []

for year in test_years:
    train = X.index[X.index.year < year]
    test = X.index[X.index.year == year]

    # Naive: next week's volatility = this week's
    fc_naive.append(X.loc[test, "log_rv_1w"])

    # Linear regression on the four inputs
    reg = LinearRegression().fit(X.loc[train], y.loc[train])
    fc_reg.append(pd.Series(reg.predict(X.loc[test]), index=test))

    # GARCH(1,1) on the daily factor returns. Parameters are estimated on data before
    # the test year (last_obs); forecasts then use each day's actual conditional state.
    # Returns are scaled by 1000 so the optimiser works on numbers near 1.
    scale = 1000
    d = daily[daily.index.year <= year] * scale
    res = arch_model(d, vol="GARCH", p=1, q=1).fit(last_obs=f"{year}-01-01", disp="off")
    f = res.forecast(horizon=5, start=f"{year}-01-01", reindex=True)
    # Average forecast variance over the next 5 days, annualised, back to our units
    var_week = f.variance.mean(axis=1) / scale**2
    vol_ann = np.sqrt(var_week * 252)
    # The forecast made on the last trading day on or before each Friday
    g = np.log(vol_ann).reindex(test, method="ffill")
    fc_garch.append(g)

    # Gaussian process on the last 5 years of the training window (standardised inputs).
    # It returns a forecast AND how unsure it is (a standard deviation).
    recent = train[-260:]
    scaler = StandardScaler().fit(X.loc[recent])
    kernel = (ConstantKernel() * RBF(length_scale=[1.0] * X.shape[1], length_scale_bounds=(0.01, 1000.0))
              + WhiteKernel())
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=helpers.SEED)
    gp.fit(scaler.transform(X.loc[recent]), y.loc[recent])
    mean, std = gp.predict(scaler.transform(X.loc[test]), return_std=True)
    fc_gp.append(pd.Series(mean, index=test))
    fc_gp_std.append(pd.Series(std, index=test))
    print(year, "done")

forecasts = pd.DataFrame({
    "naive": pd.concat(fc_naive), "regression": pd.concat(fc_reg),
    "garch": pd.concat(fc_garch), "gp": pd.concat(fc_gp),
})
gp_std = pd.concat(fc_gp_std)
realised = y.loc[forecasts.index]
print("\nForecasts:", forecasts.shape)

## 5. Forecast accuracy

In [ ]:
# Mean squared error of log volatility, and the gain over the naive forecast
# (positive = better than naive; the same score the direction model failed on).
mse = ((forecasts.sub(realised, axis=0)) ** 2).mean()
r2_vs_naive = 1 - mse / mse["naive"]

summary = pd.DataFrame({"MSE of log vol": mse, "R2 vs naive": r2_vs_naive}).round(3)
print(summary.to_string())

ax = np.exp(realised)["2019":"2021"].plot(figsize=(10, 4), label="realised", color="black")
np.exp(forecasts["gp"])["2019":"2021"].plot(ax=ax, label="Gaussian process")
np.exp(forecasts["garch"])["2019":"2021"].plot(ax=ax, label="GARCH", alpha=0.7)
ax.legend(); ax.set_title("Forecast vs realised volatility around 2020");

## 6. Market regimes: GMM, compared with k-means

In [ ]:
# Each week is described by two numbers: how wild the factor is (log rv) and how scared
# the market is (log VIX). The carry is deliberately left out: it describes the return on
# offer, not the risk state, and including it makes the groups split on the post-2020
# interest rate era instead of on risk. GMM groups the weeks walk-forward: each year is
# labelled by a model fitted only on earlier years. The number of groups (2 or 3) is chosen
# by BIC, a score that rewards fit but punishes complexity.
Z = pd.DataFrame({"log_rv": np.log(rv), "log_vix": np.log(weekly["VIX"])}).dropna()

stress_gmm, stress_km, chosen_k = [], [], []
for year in test_years:
    train, test = Z.index[Z.index.year < year], Z.index[Z.index.year == year]
    sc = StandardScaler().fit(Z.loc[train])
    Ztr, Zte = sc.transform(Z.loc[train]), sc.transform(Z.loc[test])

    gmms = {k: GaussianMixture(k, random_state=helpers.SEED).fit(Ztr) for k in (2, 3)}
    k = min(gmms, key=lambda k: gmms[k].bic(Ztr))
    chosen_k.append(k)
    gmm = gmms[k]
    # The stress regime is the group with the highest average volatility in training
    vol_by_group = pd.Series(np.log(rv).loc[train].values).groupby(gmm.predict(Ztr)).mean()
    stress_gmm.append(pd.Series(gmm.predict(Zte) == vol_by_group.idxmax(), index=test))

    km = KMeans(2, n_init=10, random_state=helpers.SEED).fit(Ztr)
    vol_by_cluster = pd.Series(np.log(rv).loc[train].values).groupby(km.predict(Ztr)).mean()
    stress_km.append(pd.Series(km.predict(Zte) == vol_by_cluster.idxmax(), index=test))

stress = pd.concat(stress_gmm).rename("stress")
stress_k = pd.concat(stress_km)
print("Groups chosen by BIC each year:", chosen_k)
print(f"Share of stress weeks (GMM): {stress.mean():.0%}")
print(f"GMM and k-means give the same label in {(stress == stress_k).mean():.0%} of weeks")
print("\nStress weeks per year:")
print(stress.groupby(stress.index.year).sum().to_string())

## 7. The two FX overlays

In [ ]:
# Both overlays run on top of the portfolio, sized by a 2% a year risk budget.
# Signals are known at the Friday close of week t and earn week t+1 (shift by one).

trend_52w = (1 + factor).rolling(52).apply(np.prod, raw=True) - 1
trend_sign = np.sign(trend_52w)

# Overlay 1, the trend rule: direction from the 52-week trend, fixed exposure.
# The factor moved about 5% a year historically, so 2% / 5% = 0.40 of the portfolio.
pos_trend = (trend_sign * 0.40).loc[forecasts.index]

# Overlay 2, the model: in calm weeks hold the carry (long); in stress weeks follow the trend.
# The size uses the Gaussian process forecast PLUS one standard deviation (its cautious
# estimate), so the position shrinks when volatility is high OR the model is unsure.
vol_cautious = np.exp(forecasts["gp"] + gp_std)
direction = pd.Series(1.0, index=forecasts.index).where(~stress, trend_sign.loc[forecasts.index])
pos_model = direction * (0.02 / vol_cautious).clip(upper=2.0)

# Weekly overlay returns, after costs (3 bp per unit of position change)
overlay = pd.DataFrame(index=forecasts.index)
for name, pos in [("trend", pos_trend), ("model", pos_model)]:
    held = pos.shift(1)
    cost = helpers.COST_BP / 1e4 * pos.diff().abs().fillna(0)
    overlay[name] = (held * factor.loc[held.index] - cost).fillna(0)

print("Average exposure: trend", f"{pos_trend.abs().mean():.2f},",
      "model", f"{pos_model.abs().mean():.2f}, cap 2.0")
print("Weeks short: trend", f"{(pos_trend < 0).mean():.0%},",
      "model", f"{(pos_model < 0).mean():.0%}")

## 8. The four portfolios in CHF

In [ ]:
# A Swiss investor with 100,000 CHF in the S&P 500, from 2010 to 2023.
# His CHF return combines the index move and the dollar's move against the franc.
idx = overlay.index
sp_usd = weekly["sp500_ret"].loc[idx]
usd_chf = weekly["ret_USDCHF"].loc[idx]          # +1% = the dollar gains 1% vs the franc

portfolios = pd.DataFrame({
    "Unhedged": (1 + sp_usd) * (1 + usd_chf) - 1,
    # Hedging the dollar costs (roughly) the rate difference between CHF and USD
    "Hedged": sp_usd + (weekly["CHF_3M"] - weekly["USD_3M"]).loc[idx] / 100 / 52,
    "Unhedged + trend": (1 + sp_usd) * (1 + usd_chf) - 1 + overlay["trend"],
    "Unhedged + model": (1 + sp_usd) * (1 + usd_chf) - 1 + overlay["model"],
})

(1 + portfolios).cumprod().mul(100_000).plot(figsize=(10, 4),
    title="100,000 CHF invested in the S&P 500, 2010 to 2023");

## 9. Crisis table and performance table

In [ ]:
# What each portfolio lost, in CHF, in the crises inside the test window.
# (2008 is before 2010: the models need training years first. The factor's 2008 behaviour
# is shown from the full sample in the figures notebook.)
crises = {
    "2011 summer (euro crisis)": ("2011-07-01", "2011-09-30"),
    "2015 SNB shock (January)": ("2015-01-01", "2015-01-31"),
    "2018 Q4 selloff": ("2018-10-01", "2018-12-31"),
    "2020 Covid crash (Feb-Mar)": ("2020-02-15", "2020-03-31"),
    "2022 full year": ("2022-01-01", "2022-12-31"),
}
rows = {}
for name, (a, b) in crises.items():
    rows[name] = ((1 + portfolios.loc[a:b]).prod() - 1) * 100_000
crisis_table = pd.DataFrame(rows).T.round(0)
print("Change of a 100,000 CHF portfolio, in CHF:")
print(crisis_table.to_string())

# The worst 16% of quarters for the unhedged investor
quarterly = (1 + portfolios).resample("QE").prod() - 1
worst = quarterly["Unhedged"] <= quarterly["Unhedged"].quantile(0.16)
print(f"\nAverage quarter, in the worst {worst.sum()} quarters of the unhedged portfolio (CHF):")
print((quarterly[worst].mean() * 100_000).round(0).to_string())

# Performance over the whole test window
perf = pd.DataFrame({
    "return p.a.": (1 + portfolios).prod() ** (52 / len(portfolios)) - 1,
    "volatility": portfolios.std() * np.sqrt(52),
    "Sharpe": portfolios.apply(helpers.sharpe_ratio),
    "worst loss": portfolios.apply(helpers.max_drawdown),
}).round(3)
print("\n2010 to 2023:")
print(perf.to_string())

pd.concat([overlay, stress, pos_trend.rename("pos_trend"), pos_model.rename("pos_model"),
           portfolios], axis=1).to_csv(f"{helpers.PROCESSED}/overlay.csv")
print("\nSaved overlay.csv")

## 10. Checks

In [ ]:
# No holdout data anywhere
for obj in (forecasts, stress, overlay, portfolios):
    assert obj.index.max() < pd.Timestamp(helpers.HOLDOUT_START), "Holdout data used"

# Forecasts exist for every test week and are positive volatilities
assert forecasts.notna().all().all(), "Missing forecasts"
assert (np.exp(forecasts) > 0).all().all()

# Positions respect the cap
assert pos_model.abs().max() <= 2.0 + 1e-9, "Position cap exceeded"

# The stress regime exists and is a minority of weeks
assert 0 < stress.mean() < 0.5, "Stress regime share looks wrong"

# Signals are lagged: the overlay return of week t uses the position decided at t-1
assert overlay.index.equals(forecasts.index)

print("All checks passed")